# Knee Bone Age — GPU training

Run top to bottom. Set **Runtime → Change runtime type → GPU** first.

## What we are trying to fix

The best local run reached 2.37 y test MAE against a 3.50 y mean-age baseline — so it
learned something, but two measurements say it is underfitting rather than near its ceiling:

| | local v3 | what good looks like |
|---|---|---|
| predicted-on-true **slope** | **0.586** | 1.0 |
| predicted range (true 3.2–17.9 y) | 5.0–15.7 y | ~3–18 y |
| within 1 year | 17% | 60%+ |

A slope of 0.59 means the model hedges toward the middle: it over-predicted a 3.2-year-old
by +2.6 y and under-predicted a 17.9-year-old by −6.0 y.

The likeliest cause is resolution. Local runs were stuck at 16×96×96, where a 96-pixel
field of view over a ~130 mm knee is ~1.35 mm/pixel and 16 slices means ~6 mm thick. The
growth plate is 0.2–3.2 mm thick and its thinning **is** the bone age signal — at that
sampling it is sub-voxel, so the network cannot see the feature it needs.

This notebook trains at **32×192×192** (~0.7 mm in-plane, 3 mm slices, so the physis spans
several voxels) on **400** phantoms for **40** epochs.

In [ ]:
!nvidia-smi
import torch; print('CUDA available:', torch.cuda.is_available())

In [ ]:
# 1. Get the code
!git clone https://github.com/anshppatel4-crypto/knee-bone-age-ai.git
%cd knee-bone-age-ai
!pip install -q monai pydicom scikit-image

In [ ]:
# 2. Generate phantoms. Measured at 8.4 s/scan on one core, so this is CPU-bound:
#    ~30 min on a 2-core Colab box, less if you drew a bigger one.
#    More data is the cheapest remaining win: 400 beats the 240 used locally.
import os, subprocess, time

TOTAL = 400
cores = os.cpu_count() or 2
per_job = -(-TOTAL // cores)  # ceiling, so jobs cover TOTAL between them
print(f'{cores} cores -> {cores} jobs x {per_job} scans; estimate {TOTAL * 8.4 / cores / 60:.0f} min')

start = time.time()
jobs = [subprocess.Popen(['python', '-m', 'src.knee_phantom', '--count', str(per_job),
                          '--out', f'data/phantom_gpu_{i:02d}', '--seed', str(31 + i),
                          '--slices', '32', '--resolution', '256'])
        for i in range(cores)]
failed = [job.wait() for job in jobs]
print(f'generation took {(time.time() - start) / 60:.1f} min; exit codes {failed}')
assert not any(failed), 'a generation job failed -- read its traceback above before training'
!ls -d data/phantom_gpu_*/knee_* | wc -l

In [ ]:
# 3. Train.
#    --workers matters as much as the GPU here: augmenting one 32x192x192 volume costs
#    0.27 s of CPU, and with workers=0 (the default) the GPU waits on it for every
#    sample. Two workers overlap that with compute.
#    Drop --batch-size to 2 if you hit CUDA out-of-memory.
import os
workers = min(4, max(2, os.cpu_count() or 2))
print('dataloader workers:', workers)

!python src/train.py --data 'data/phantom_gpu_*' --arch resnet34 --epochs 40 --batch-size 4 --lr 3e-4 --workers {workers} --input-shape 32 192 192 --output final_knee_model_gpu.pth

In [ ]:
# 4. Did it actually learn? Three checks that matter more than the headline MAE.
import numpy as np, sys
sys.path.insert(0, '.')
from torch.utils.data import DataLoader
from src.model import load_checkpoint
from src.predict import predict_scan
from src.dataset import KneeVolumeDataset
from src.train import build_catalog, split_catalog, predict_loader, regression_metrics

test = split_catalog(build_catalog(['data/phantom_gpu_*']), seed=0)[2]
model, meta = load_checkpoint('final_knee_model_gpu.pth', 'cuda')
dataset = KneeVolumeDataset(test, input_shape=tuple(meta['input_shape']), augment=False)

pred, targ = predict_loader(model, DataLoader(dataset, batch_size=4), 'cuda', tta=True)
metrics = regression_metrics(pred, targ)

# (a) Slope. This is the one that was 0.586 locally. MAE can improve while the model
#     still just hedges toward the mean, and slope is what separates the two.
print(f"MAE {metrics['mae']:.3f}y | within 1y {metrics['within_1y']:.0%} | "
      f"slope {metrics['slope']:.3f} (want ~1.0) | corr {metrics['corr']:.3f}")
print(f"predicted {metrics['pred_range'][0]:.1f}-{metrics['pred_range'][1]:.1f}y over a true "
      f"{metrics['true_range'][0]:.1f}-{metrics['true_range'][1]:.1f}y span")

# (b) Error by age band. Local runs were worst at 17+ (4.6 y), where the physis has
#     fused and the remaining cue is marrow conversion.
bands = np.digitize(targ, [6, 9, 12, 15, 17])
for band, label in enumerate(['<6', '6-9', '9-12', '12-15', '15-17', '17+']):
    mask = bands == band
    if mask.any():
        print(f'  {label:>6}: n={mask.sum():3d}  MAE {np.abs(pred[mask] - targ[mask]).mean():.3f}y')

# (c) Sex must matter. The phantom defines maturity as age + 1.8 y for girls, so for a
#     fixed image, being told "female" should return an age ~1.8 y LOWER than "male".
#     Local runs measured 0.02 y, which meant sex was being ignored outright.
deltas = []
for i in range(min(15, len(dataset))):
    volume = dataset[i]['image'].numpy()[0]
    deltas.append(predict_scan(model, volume, 'm', 'cuda')['bone_age']
                  - predict_scan(model, volume, 'f', 'cuda')['bone_age'])
print(f'sex delta (M-F): {np.mean(deltas):+.3f} years (want ~+1.8)')

In [ ]:
# 5. Download the checkpoint and metrics
from google.colab import files
files.download('final_knee_model_gpu.pth')
files.download('final_knee_model_gpu_metrics.json')

## Reading the result

Read these in order — each one can invalidate the one above it.

1. **Slope ~1.0.** Locally 0.586. If MAE improved but slope is still ~0.6, the model got
   better at hedging, not at reading anatomy, and the headline MAE is flattering it.
2. **Test MAE vs the printed baseline.** Training prints the "always guess the mean age"
   number. Beating it by a wide margin is the minimum bar, not a result.
3. **Sex delta ~+1.8 y.** The model is using skeletal maturity *and* sex, as a radiologist
   does. Near zero means it is ignoring a label the phantom definitely encodes.
4. **Train loss well below val MAE.** Finally fitting. Locally they were about equal,
   which is the signature of underfitting.

If slope is still low and train loss is still tracking val MAE, the limit is capacity or
schedule, not resolution: try `--epochs 80`, or `--arch resnet18` with `--batch-size 8`
(a smaller trunk on 400 scans can beat a larger one).

### The caveat that does not go away

Every number here is accuracy **on synthetic phantoms**, measured against labels the
phantom generator wrote itself. It tells you the network can extract maturity from
anatomy the generator drew. It does **not** tell you the error on a real patient, and
the sex model in particular is a flat 1.8-year offset, whereas real knee maturation
diverges by sex in a way that varies with age. Validating on real scans is a separate
job from this one.